# 03. Feature Engineering Data Preparation

이 노트북은 파생변수 생성에 사용하기 전 외부 입지 데이터를 정제한다.

## 정제 대상

- 병원정보서비스 2025년 3월, 6월, 9월, 12월 파일에서 서울 소재 상급종합/종합병원만 비교
- 서울시 역사마스터 정보는 별도 필터링 없이 원본 그대로 사용
- 서울시 대규모점포 인허가 정보에서 영업 중인 대형마트만 필터링

원본 파일은 `data/external/`에 유지하고, 정제 결과는 `data/interim/`에 저장한다.

## 1. 라이브러리 및 경로 설정

In [1]:
from pathlib import Path
import sys
import importlib

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
src_path = str(PROJECT_ROOT / 'src')
if src_path not in sys.path:
    sys.path.append(src_path)

import features
import geocode_naver
importlib.reload(features)
importlib.reload(geocode_naver)

from features import (
    CBD_CENTERS,
    add_accessibility_features,
    clean_large_marts,
    compare_hospital_snapshots,
    find_external_file,
    get_hospital_quarter,
    load_target_hospitals,
    normalize_filename,
)
from geocode_naver import geocode_unique_addresses, merge_coordinates, test_naver_geocoding

EXTERNAL_DIR = PROJECT_ROOT / 'data/external'
INTERIM_DIR = PROJECT_ROOT / 'data/interim'
PROCESSED_DIR = PROJECT_ROOT / 'data/processed'
INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

EXTERNAL_DIR


PosixPath('/Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/data/external')

## 2. 병원정보서비스 분기별 비교

상급종합병원과 종합병원은 개수가 많지 않고 시점별 변화가 가격 접근성 변수에 영향을 줄 수 있으므로, 3월·6월·9월·12월 스냅샷을 먼저 비교한다.

In [2]:
hospital_files = sorted(
    [path for path in EXTERNAL_DIR.glob('*.csv') if '병원정보서비스' in normalize_filename(path.name)],
    key=get_hospital_quarter,
)

hospital_by_month = {
    get_hospital_quarter(path): load_target_hospitals(path)
    for path in hospital_files
}

hospital_summary, hospital_diffs = compare_hospital_snapshots(hospital_by_month)
hospital_summary

,snapshot_month,total_count,tertiary_count,general_count
0,03,58,14,44
1,06,59,14,45
2,09,59,14,45
3,12,59,14,45


In [3]:
hospital_diffs

,base_month,compare_month,change_type,hospital_name,hospital_type,gu,address
0,03,06,added,서울현대병원,종합병원,강북구,"서울특별시 강북구 도봉로 374, (번동, 서울현대병원)"
1,03,09,added,서울현대병원,종합병원,강북구,"서울특별시 강북구 도봉로 374, (번동, 서울현대병원)"
2,03,12,added,서울현대병원,종합병원,강북구,"서울특별시 강북구 도봉로 374, (번동, 서울현대병원)"


비교 결과 2025년 3월은 58개, 6월·9월·12월은 59개이다. 6월 이후 스냅샷은 서로 동일하며, 3월 대비 `서울현대병원` 1개 종합병원이 추가되어 있다.

따라서 연간 분석용 병원 접근성 변수에는 가장 최근 스냅샷인 12월 자료를 대표 병원 목록으로 사용한다.

In [4]:
selected_hospital_month = '12'
seoul_hospitals = hospital_by_month[selected_hospital_month].copy()
seoul_hospitals.head()

,snapshot_month,hospital_id,hospital_name,hospital_type,sido,gu,address,phone,opened_date,doctor_count,longitude,latitude
0,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,삼성서울병원,상급종합,서울,강남구,"서울특별시 강남구 일원로 81, (일원동, 삼성의료원)",02-3410-2114,1994.6.13,1285,127.085151,37.488298
1,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,연세대학교의과대학 강남세브란스병원,상급종합,서울,강남구,"서울특별시 강남구 언주로 211, 강남세브란스병원 (도곡동)",02-2019-3114,1983.4.4,504,127.046268,37.492930
2,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,건국대학교병원,상급종합,서울,광진구,"서울특별시 광진구 능동로 120-1, (화양동)",1588-1533,1982.11.16,357,127.071828,37.540376
3,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,고려대학교의과대학부속구로병원,상급종합,서울,구로구,"서울특별시 구로구 구로동로 148, 고려대부속구로병원 (구로동)",02-2626-1114,1983.8.31,497,126.884870,37.492052
4,12,JDQ4MTg4MSM1MSMkMSMkMCMkODkkMzgxMzUxIzExIyQxIy...,경희대학교병원,상급종합,서울,동대문구,"서울특별시 동대문구 경희대로 23, (회기동)",02-958-8114,1971.10.5,394,127.051852,37.594119


## 3. 역사마스터 원본 확인

역사마스터 데이터는 서울시 제공 파일을 그대로 사용한다. 별도 필터링 없이 역사 ID, 역사명, 호선, 위도, 경도 컬럼을 확인한다.

추후 거리 파생변수 계산 단계에서 필요하면 역명·호선 중복이나 환승역 처리 기준만 별도로 정한다.

In [5]:
station_path = find_external_file(EXTERNAL_DIR, '역사마스터')
station_master = pd.read_csv(station_path, encoding='cp949')

station_master.shape, station_master.head()

((783, 5),
    역사_ID 역사명          호선        위도         경도
 0   9010  동탄  수도권 광역급행철도  37.20034  127.09569
 1   9009  구성  수도권 광역급행철도  37.29913  127.10389
 2   9008  성남  수도권 광역급행철도  37.39467  127.12058
 3   9007  수서  수도권 광역급행철도  37.48637  127.10161
 4   9006  삼성  수도권 광역급행철도  37.50887  127.06324)

In [6]:
station_master['호선'].value_counts().head(15)

호선
5호선      56
7호선      53
2호선      50
6호선      39
경부선      39
분당선      34
3호선      34
인천1호선    33
경원선      32
인천2호선    27
경의중앙선    27
4호선      26
9호선      25
중앙선      21
경인선      20
Name: count, dtype: int64

## 4. 영업 중인 대형마트 필터링

대규모점포 인허가 정보에서 `업태구분명 == 대형마트`, `영업상태명 == 영업/정상`, `상세영업상태명 == 정상영업`인 행만 사용한다.

In [7]:
large_store_path = find_external_file(EXTERNAL_DIR, '대규모점포')
large_marts = clean_large_marts(large_store_path)

large_marts.shape, large_marts.head()

((64, 10),
        store_id   store_name store_type business_status  \
 0  2.006320e+18   (주)이마트 역삼점       대형마트           영업/정상   
 1  2.005320e+18    (주)이마트천호점       대형마트           영업/정상   
 2  2.009320e+18     홈플러스 강동점       대형마트           영업/정상   
 3  2.012310e+18  (주)농협유통 미아점       대형마트           영업/정상   
 4  2.005320e+18  홈플러스(주) 강서점       대형마트           영업/정상   
 
   business_status_detail licensed_date   gu                    address  \
 0                   정상영업    2006.10.24  강남구    서울특별시 강남구 역삼로 310 (역삼동)   
 1                   정상영업     2005.3.31  강동구  서울특별시 강동구 천호대로 1017 (천호동)   
 2                   정상영업     2008.11.7  강동구  서울특별시 강동구 양재대로 1571 (천호동)   
 3                   정상영업     2012.3.22  강북구  서울특별시 강북구 도봉로33길 18 (미아동)   
 4                   정상영업     2005.8.26  강서구  서울특별시 강서구  화곡로  398 (등촌동)   
 
        coord_x      coord_y  
 0  204213.6432  444113.0282  
 1  211025.9250  448495.1892  
 2  212501.3691  449282.3104  
 3  202266.4487  457691.8024  
 4  187119.9482  450

In [8]:
large_marts['gu'].value_counts(dropna=False).sort_index()

gu
강남구     1
강동구     2
강북구     1
강서구     1
관악구     1
광진구     6
구로구     3
금천구     3
노원구     2
도봉구     1
동대문구    1
마포구     5
망우로     1
서초구     4
성동구     1
성북구     7
송파구     4
양천구     1
영등포구    6
은평구     4
종로구     1
중구      2
중랑구     5
NaN     1
Name: count, dtype: int64

대형마트 좌표는 원본의 `좌표정보(X)`, `좌표정보(Y)`를 보존한다. 이 좌표는 지하철·병원 위경도와 좌표계가 다르므로, 거리 파생변수 계산 전에는 좌표계 변환 또는 주소 기반 지오코딩이 필요하다.

## 5. Naver 지오코딩으로 좌표 생성

아파트 거래 데이터의 `full_road_address` 고유값만 Naver Geocoding API에 요청해 좌표를 생성한다. 전체 거래 행은 77,359건이지만 고유 도로명 주소는 5,769개이므로, 중복 주소를 제거한 뒤 좌표를 구하고 다시 원본 거래 데이터에 병합한다.

대형마트도 반경 1km 카운트를 계산하려면 위경도가 필요하므로, 정제한 대형마트 주소도 같은 방식으로 좌표를 생성한다.

`.env`에는 다음 값을 설정한다.

```text
NAVER_CLIENT_ID=...
NAVER_CLIENT_SECRET=...
```

In [9]:
apt_path = INTERIM_DIR / 'seoul_apt_trade_2025_basic_cleaned.csv'
apt_geocoded_path = INTERIM_DIR / 'seoul_apt_address_geocoded.csv'
apt_with_coordinates_path = PROCESSED_DIR / 'seoul_apt_trade_2025_with_coordinates.csv'
large_mart_address_path = INTERIM_DIR / 'seoul_large_mart_addresses.csv'
large_mart_geocoded_path = INTERIM_DIR / 'seoul_large_mart_addresses_geocoded.csv'
large_mart_with_coordinates_path = INTERIM_DIR / 'seoul_large_marts_geocoded.csv'
features_output_path = PROCESSED_DIR / 'seoul_apt_trade_2025_features.csv'

apt_df = pd.read_csv(apt_path, encoding='utf-8-sig')
unique_address_count = apt_df['full_road_address'].nunique(dropna=True)
print(f'전체 거래 행: {len(apt_df):,}')
print(f'고유 도로명 주소: {unique_address_count:,}')

전체 거래 행: 77,359
고유 도로명 주소: 5,769


In [10]:
# 키 값을 노출하지 않고 Naver Geocoding 인증 상태만 확인한다.
# status_code가 200이면 인증 성공이다. 실패해도 노트북 실행은 멈추지 않는다.
try:
    auth_test_result = test_naver_geocoding()
    print(auth_test_result)
except Exception as exc:
    auth_test_result = {'status_code': 'failed', 'error': str(exc)}
    print('Naver 지오코딩 테스트 실패:', exc)
    print('인터넷 연결, API 키, 또는 Naver Maps 서비스 상태를 확인하세요.')


{'client_id': 'fw03...zzm2', 'client_secret': 'cjB3...EsWS', 'status_code': 200, 'body': {'status': 'OK', 'meta': {'totalCount': 1, 'page': 1, 'count': 1}, 'addresses': [{'roadAddress': '서울특별시 중구 세종대로 110 서울특별시청', 'jibunAddress': '서울특별시 중구 태평로1가 31 서울특별시청', 'englishAddress': '110, Sejong-daero, Jung-gu, Seoul, Republic of Korea', 'addressElements': [{'types': ['SIDO'], 'longName': '서울특별시', 'shortName': '서울특별시', 'code': ''}, {'types': ['SIGUGUN'], 'longName': '중구', 'shortName': '중구', 'code': ''}, {'types': ['DONGMYUN'], 'longName': '태평로1가', 'shortName': '태평로1가', 'code': ''}, {'types': ['RI'], 'longName': '', 'shortName': '', 'code': ''}, {'types': ['ROAD_NAME'], 'longName': '세종대로', 'shortName': '세종대로', 'code': ''}, {'types': ['BUILDING_NUMBER'], 'longName': '110', 'shortName': '110', 'code': ''}, {'types': ['BUILDING_NAME'], 'longName': '서울특별시청', 'shortName': '서울특별시청', 'code': ''}, {'types': ['LAND_NUMBER'], 'longName': '31', 'shortName': '31', 'code': ''}, {'types': ['POSTAL_CODE'], 'l

In [ ]:
# 처음 실행할 때는 RUN_NAVER_GEOCODING=True, GEOCODE_LIMIT=10으로 소량 테스트한다.
# 테스트 성공 후 GEOCODE_LIMIT=None으로 바꾸면 전체 고유 주소를 처리한다.
RUN_NAVER_GEOCODING = True
GEOCODE_LIMIT = None
GEOCODE_FORCE = True

if RUN_NAVER_GEOCODING:
    geocode_unique_addresses(
        input_path=apt_path,
        output_path=apt_geocoded_path,
        address_col='full_road_address',
        sleep_seconds=0.1,
        limit=GEOCODE_LIMIT,
        force=GEOCODE_FORCE,
        max_workers=5,
        max_retries=3,
    )
    merge_coordinates(
        apartment_path=apt_path,
        geocoded_path=apt_geocoded_path,
        output_path=apt_with_coordinates_path,
        address_col='full_road_address',
    )
else:
    print('RUN_NAVER_GEOCODING=False: API 호출은 실행하지 않습니다.')

In [ ]:
large_marts[['address']].dropna().drop_duplicates().to_csv(
    large_mart_address_path, index=False, encoding='utf-8-sig'
)

if RUN_NAVER_GEOCODING:
    geocode_unique_addresses(
        input_path=large_mart_address_path,
        output_path=large_mart_geocoded_path,
        address_col='address',
        sleep_seconds=0.1,
        limit=None,
        force=GEOCODE_FORCE,
        max_workers=5,
        max_retries=3,
    )

if large_mart_geocoded_path.exists():
    large_mart_coordinates = pd.read_csv(large_mart_geocoded_path, encoding='utf-8-sig')
    large_marts_for_features = large_marts.merge(
        large_mart_coordinates[['address', 'latitude', 'longitude', 'geocode_status']],
        on='address',
        how='left',
    )
    large_marts_for_features.to_csv(large_mart_with_coordinates_path, index=False, encoding='utf-8-sig')
    print(f'대형마트 좌표 보유: {large_marts_for_features["latitude"].notna().sum():,}/{len(large_marts_for_features):,}')
else:
    large_marts_for_features = None
    print('대형마트 지오코딩 결과가 아직 없습니다.')

## 6. 거리 기반 파생변수 생성

아파트 좌표가 붙은 `seoul_apt_trade_2025_with_coordinates.csv`가 있으면 다음 파생변수를 계산해 `data/processed/seoul_apt_trade_2025_features.csv`로 저장한다.

- `nearest_subway_distance_km`: 아파트 좌표와 전체 역 좌표 사이의 최단 거리
- `distance_to_cbd_km`, `distance_to_ybd_km`, `distance_to_gbd_km`: 3대 업무지구별 거리
- `nearest_business_district_distance_km`: CBD/YBD/GBD 중심 좌표 중 최단 거리
- `nearest_business_district`: 가장 가까운 업무지구 코드
- `hospital_count_within_1km`: 반경 1km 이내 상급종합병원/종합병원 수
- `nearest_hospital_distance_km`: 상급종합병원/종합병원까지의 최단 거리
- `large_mart_count_within_1km`: 반경 1km 이내 대형마트 수

중심업무지구 좌표는 `src/features.py`의 `CBD_CENTERS` 상수에 고정해 사용한다.

In [ ]:
pd.DataFrame(CBD_CENTERS).T

,name,description,address,latitude,longitude
CBD,도심권,시청/광화문 일대,서울특별시 중구 세종대로 110,37.5665,126.978
YBD,여의도권,여의도 일대,서울특별시 영등포구 의사당대로 1,37.5259,126.9209
GBD,강남권,강남역 사거리 일대,서울특별시 강남구 강남대로 396,37.4979,127.0276


In [ ]:
stations_for_features = station_master.rename(
    columns={'역사_ID': 'station_id', '역사명': 'station_name', '호선': 'line', '위도': 'latitude', '경도': 'longitude'}
)[['station_id', 'station_name', 'line', 'latitude', 'longitude']]

hospitals_for_features = seoul_hospitals.dropna(subset=['latitude', 'longitude']).copy()

if large_mart_with_coordinates_path.exists():
    large_marts_for_features = pd.read_csv(large_mart_with_coordinates_path, encoding='utf-8-sig')
    large_marts_for_features = large_marts_for_features.dropna(subset=['latitude', 'longitude'])
else:
    large_marts_for_features = None

print(f'역 좌표: {len(stations_for_features):,}')
print(f'병원 좌표: {len(hospitals_for_features):,}')
print('대형마트 좌표:', '없음' if large_marts_for_features is None else f'{len(large_marts_for_features):,}')

역 좌표: 783
병원 좌표: 59
대형마트 좌표: 62


In [ ]:
if apt_with_coordinates_path.exists():
    apt_with_coordinates = pd.read_csv(apt_with_coordinates_path, encoding='utf-8-sig')
    coordinate_count = apt_with_coordinates[['latitude', 'longitude']].dropna().shape[0]
    print(f'아파트 좌표 보유: {coordinate_count:,}/{len(apt_with_coordinates):,}')

    if coordinate_count == 0:
        print('좌표가 하나도 없어 거리 기반 파생변수를 생성하지 않습니다.')
        print('Naver 지오코딩 결과의 geocode_status/error_message를 먼저 확인하세요.')
    else:
        apt_features = add_accessibility_features(
            apt_with_coordinates,
            stations=stations_for_features,
            hospitals=hospitals_for_features,
            large_marts=large_marts_for_features,
        )
        apt_features.to_csv(features_output_path, index=False, encoding='utf-8-sig')
        print(f'saved: {features_output_path}')
        print(f'최종 데이터: {apt_features.shape[0]:,}행 x {apt_features.shape[1]:,}열')
        preview_columns = [
            'apartment_name',
            'full_road_address',
            'latitude',
            'longitude',
            'nearest_subway_distance_km',
            'nearest_business_district',
            'nearest_business_district_distance_km',
            'hospital_count_within_1km',
            'nearest_hospital_distance_km',
        ]
        display(apt_features.dropna(subset=['latitude', 'longitude'])[preview_columns].head())
else:
    print(f'아파트 좌표 파일이 아직 없습니다: {apt_with_coordinates_path}')
    print('위 Naver 지오코딩 셀에서 RUN_NAVER_GEOCODING=True로 실행한 뒤 다시 실행하세요.')


아파트 좌표 보유: 77,334/77,359
saved: /Users/cheong-kyumin/HSU/데이터마이닝/dm-apt-predict/dm-apt-predict/data/processed/seoul_apt_trade_2025_features.csv
최종 데이터: 77,359행 x 31열


,apartment_name,full_road_address,latitude,longitude,nearest_subway_distance_km,nearest_business_district,nearest_business_district_distance_km,hospital_count_within_1km,nearest_hospital_distance_km
0,왕십리KCC스위첸,서울특별시 성동구 무학봉길 35,37.560960,127.026967,0.432088,CBD,4.359801,1,0.470749
1,대성아파트,서울특별시 종로구 사직로 21,37.572812,126.962566,0.467477,CBD,1.530684,3,0.521502
2,남산센트럴자이,서울특별시 중구 퇴계로 235,37.562514,126.997824,0.352045,CBD,1.802663,1,0.866817
3,현대,서울특별시 성동구 살곶이길 50,37.569933,127.042494,0.351577,CBD,5.697082,0,1.151700
4,남산센트럴자이,서울특별시 중구 퇴계로 235,37.562514,126.997824,0.352045,CBD,1.802663,1,0.866817


## 7. 정제 결과 저장

In [ ]:
hospital_summary.to_csv(INTERIM_DIR / 'hospital_2025_quarter_comparison.csv', index=False, encoding='utf-8-sig')
hospital_diffs.to_csv(INTERIM_DIR / 'hospital_2025_quarter_differences.csv', index=False, encoding='utf-8-sig')
seoul_hospitals.to_csv(INTERIM_DIR / 'seoul_hospitals_tertiary_general_2025_12.csv', index=False, encoding='utf-8-sig')
large_marts.to_csv(INTERIM_DIR / 'seoul_large_marts_cleaned.csv', index=False, encoding='utf-8-sig')

sorted(path.name for path in INTERIM_DIR.glob('*.csv'))

['hospital_2025_quarter_comparison.csv',
 'hospital_2025_quarter_differences.csv',
 'seoul_apt_address_geocoded.csv',
 'seoul_apt_trade_2025_basic_cleaned.csv',
 'seoul_hospitals_tertiary_general_2025_12.csv',
 'seoul_large_mart_addresses.csv',
 'seoul_large_mart_addresses_geocoded.csv',
 'seoul_large_marts_cleaned.csv',
 'seoul_large_marts_geocoded.csv',
 'seoul_subway_stations_cleaned.csv']